# Observability & Tracing with Arize Phoenix

Every episode so far has trusted a single printed answer. **Observability** shows what actually happened _inside_ a query — every embedding call, every retrieval step, every LLM completion, in order, with timing. This uses [Arize Phoenix](https://arize.com/docs/phoenix), self-hosted locally with no signup or API key, instrumented via OpenTelemetry.


**Step 1 — Setup.** Silence noisy gRPC/warning output, load API keys, and set the default LLM/embedding model — same pattern as every earlier episode.


In [1]:
import logging
import os
import warnings

# Phoenix uses gRPC under the hood for span export, which can be chatty — turn
# it down to errors only, and silence general Python warnings too.
os.environ["GRPC_VERBOSITY"] = "ERROR"
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Quiet the usual INFO-level noise from httpx and llama_index.
for noisy_logger in ("httpx", "llama_index"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Loads keys like OPENAI_API_KEY from .env into os.environ.
load_dotenv()

Settings.llm = OpenAI(model="gpt-4.1-nano")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

**Step 2 — Launch Phoenix and start tracing.** `px.launch_app()` starts a local Phoenix server; `LlamaIndexInstrumentor().instrument()` is the one call that hooks LlamaIndex's internals into OpenTelemetry so every future call gets traced automatically.


In [2]:
import phoenix as px
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor
from phoenix.otel import register

# Starts a local Phoenix web server + backing store, with a UI to browse traces.
session = px.launch_app()
print(f"Phoenix UI running at {session.url} — open this in a browser to see traces live")

# register() wires up an OpenTelemetry tracer that ships spans to this Phoenix session.
tracer_provider = register(
    endpoint=f"{session.url}v1/traces",
    project_name="anime-rag-demo",
    verbose=False,
)
# One-time instrumentation call: from here on, every LlamaIndex call in this
# kernel emits trace spans automatically, with no changes to the RAG code itself.
LlamaIndexInstrumentor().instrument(tracer_provider=tracer_provider)

boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
Phoenix UI running at http://localhost:6006/ — open this in a browser to see traces live


**Step 3 — Run normal queries.** Nothing here is different from earlier episodes — build the index, ask two questions — but now every step underneath is being traced in the background.


In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

# The same load -> chunk -> embed -> index -> query engine pipeline as earlier episodes —
# every call below is now silently emitting trace spans thanks to Step 2.
documents = SimpleDirectoryReader("data/sample_docs").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

for question in (
    "What is Naruto's signature technique?",
    "How many Dragon Balls are needed to summon Shenron?",
    "Tanjiro Kamado uses water breathing and he uses another breathing technique as well, provide the name of it as well?",
):
    response = query_engine.query(question)
    print(f"Q: {question}\nA: {response}\n")

Q: What is Naruto's signature technique?
A: Naruto's signature technique is the Rasengan, a swirling ball of concentrated chakra.

Q: How many Dragon Balls are needed to summon Shenron?
A: Seven Dragon Balls are needed to summon Shenron.

Q: Tanjiro Kamado uses water breathing and he uses another breathing technique as well, provide the name of it as well?
A: Hinokami Kagura, also known as Sun Breathing



**Step 4 — Pull the captured traces back programmatically.** Instead of only viewing traces in the Phoenix UI, fetch them as a dataframe via the Phoenix client and list every distinct internal step LlamaIndex performed for these two queries.


In [4]:
import time

from phoenix.client import Client

time.sleep(3)  # give the trace exporter a moment to flush spans to Phoenix
client = Client(base_url=session.url)
# One row per captured span (a single traced operation), across both queries above.
spans_df = client.spans.get_spans_dataframe(project_name="anime-rag-demo")

print(f"Captured {len(spans_df)} spans across both queries. Distinct span types:\n")
for span_name in sorted(spans_df["name"].unique()):  # each name is one kind of internal operation
    print(f"- {span_name}")

Captured 36 spans across both queries. Distinct span types:

- CompactAndRefine.get_response
- CompactAndRefine.synthesize
- DefaultRefineProgram.__call__
- OpenAI.chat
- OpenAI.predict
- OpenAIEmbedding._get_query_embedding
- OpenAIEmbedding.get_query_embedding
- OpenAIEmbedding.get_text_embedding_batch
- RetrieverQueryEngine._query
- RetrieverQueryEngine.query
- SentenceSplitter.__call__
- SentenceSplitter._parse_nodes
- SentenceSplitter.split_text_metadata_aware
- TokenTextSplitter.split_text
- VectorIndexRetriever._retrieve
- VectorIndexRetriever.retrieve


### Summary

- `LlamaIndexInstrumentor().instrument(tracer_provider=...)` is a one-time setup call — every index build and query afterward gets traced automatically, with zero changes to the actual RAG code.
- The span names above show LlamaIndex's real internal pipeline for a single `.query()` call: sentence splitting → embedding → retrieval → response synthesis → the final LLM chat completion — the exact sequence every earlier episode has been trusting implicitly.
- Open the Phoenix URL printed above in a browser (while this kernel is still running) to see the same traces as a visual, clickable timeline instead of a flat span list.
